# Wastewater respiratory pathogen analysis

This notebook inventories the downloaded wastewater datasets, inspects source schemas, and sets up a canonical long-format analysis table.

The datasets are heterogeneous across countries, so the first task is schema inspection rather than immediate merging.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from wastewater.io import find_repo_root, load_sources, inventory_raw_files, read_table, read_zip_tables, inspect_source_schemas
from wastewater.cleaning import canonical_empty_frame, normalise_column_names, add_time_features, add_log_signal, add_within_series_zscore, add_rolling_features, latest_signal_summary
from wastewater.plotting import plot_country_pathogen_timeseries

ROOT = find_repo_root(ROOT)
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
ROOT

## 1. Source catalogue

`sources.csv` is the machine-readable map of what should exist in `data/raw/`.

In [ ]:
sources = load_sources(ROOT)
display(sources)

sources.groupby(['country', 'pathogen_scope', 'level', 'format'], dropna=False).size().reset_index(name='n_sources')

## 2. Inventory raw files

Run the download workflow or `python scripts/download_all.py` before expecting every file to be present.

In [ ]:
inventory = inventory_raw_files(ROOT)
display(inventory.sort_values('path'))

if not inventory.empty:
    display(inventory.groupby('suffix', dropna=False)['size_mb'].agg(['count', 'sum']).sort_values('sum', ascending=False))

## 3. Inspect schemas

Do not merge countries until the source-specific schemas and units are understood.

In [ ]:
schemas = inspect_source_schemas(ROOT)
schema_table = pd.DataFrame([
    {
        'country': x.get('country'),
        'pathogen_scope': x.get('pathogen_scope'),
        'level': x.get('level'),
        'source_file': x.get('source_file'),
        'member': x.get('member'),
        'exists': x.get('exists'),
        'n_cols': x.get('n_cols'),
        'columns': ', '.join(map(str, x.get('columns', [])))[:300],
        'error': x.get('error', ''),
    }
    for x in schemas
])
display(schema_table)

## 4. Preview a source

Use this helper for interactive inspection. It reads ZIP members if the source is archived.

In [ ]:
def preview_source(filename: str, n: int = 5):
    path = RAW / filename
    if not path.exists():
        print(f'Missing: {path}')
        return None

    if path.suffix.lower() == '.zip':
        tables = read_zip_tables(path)
        for member, df in tables.items():
            print(f'\n--- {filename} :: {member} ---')
            print(df.shape)
            display(df.head(n))
        return tables

    df = read_table(path)
    print(df.shape)
    display(df.head(n))
    return df

# Examples after data download/extraction:
# preview_source('germany_amelag_aggregated_curve.tsv')
# preview_source('netherlands_rivm_national.csv')
# preview_source('belgium_sars_cov_2.zip')

## 5. Canonical long-format target

Aim to map each source into this schema. Absolute viral loads should not be compared across countries unless units and lab methods are compatible. Prefer within-series z-scores, percentiles, or growth rates for cross-country comparisons.

In [ ]:
canonical_columns = canonical_empty_frame().columns.tolist()
canonical_columns

## 6. Cleaning adapter template

Replace this placeholder with explicit country/source-specific mappings after schema inspection.

In [ ]:
def clean_placeholder(df: pd.DataFrame, *, country: str, pathogen: str, level: str, source_file: str) -> pd.DataFrame:
    df = normalise_column_names(df)

    date_candidates = [c for c in df.columns if c in {'date', 'datum', 'week', 'sample_date', 'sampling_date', 'date_start'}]
    value_candidates = [c for c in df.columns if any(k in c for k in ['value', 'load', 'rna', 'copies', 'concentration', 'signal'])]

    if not date_candidates or not value_candidates:
        raise ValueError(f'Need explicit mapping for {source_file}; columns={list(df.columns)}')

    date_col = date_candidates[0]
    value_col = value_candidates[0]

    return pd.DataFrame({
        'country': country,
        'source': source_file,
        'pathogen': pathogen,
        'date': pd.to_datetime(df[date_col], errors='coerce'),
        'site_id': df.get('site_id', df.get('wwtp_id', df.get('location_id', pd.NA))),
        'site_name': df.get('site_name', df.get('wwtp_name', df.get('location', pd.NA))),
        'geography_level': level,
        'region': df.get('region', df.get('area', df.get('province', pd.NA))),
        'value': pd.to_numeric(df[value_col], errors='coerce'),
        'value_unit': pd.NA,
        'population': pd.NA,
        'normalised_value': pd.NA,
        'normalised_unit': pd.NA,
        'source_file': source_file,
    }).dropna(subset=['date', 'value'])

## 7. Identify sources needing explicit mappings

In [ ]:
needs_mapping = []

for row in sources.to_dict(orient='records'):
    filename = row['filename']
    path = RAW / filename
    if not path.exists():
        needs_mapping.append({**row, 'reason': 'raw file missing'})
        continue
    if path.suffix.lower() == '.zip':
        needs_mapping.append({**row, 'reason': 'ZIP source: inspect extracted/member tables first'})
        continue
    try:
        df = read_table(path, nrows=1000)
        _ = clean_placeholder(df, country=row['country'], pathogen=row['pathogen_scope'], level=row['level'], source_file=filename)
    except Exception as exc:
        needs_mapping.append({**row, 'reason': repr(exc)})

pd.DataFrame(needs_mapping)

## 8. Build the canonical dataset

Once explicit adapters are implemented, append their outputs to `canonical_parts`.

In [ ]:
canonical_parts = []

# Example pattern:
# df = read_table(RAW / 'germany_amelag_aggregated_curve.tsv')
# canonical_parts.append(clean_germany_aggregated(df, source_file='germany_amelag_aggregated_curve.tsv'))

clean_df = pd.concat(canonical_parts, ignore_index=True) if canonical_parts else canonical_empty_frame()
clean_df.head()

## 9. Add standard features and save processed data

In [ ]:
if clean_df.empty:
    print('clean_df is empty. Add explicit source cleaning functions first.')
else:
    clean_df = (
        clean_df
        .pipe(add_time_features, date_col='date')
        .pipe(add_log_signal, value_col='value')
        .pipe(add_within_series_zscore, value_col='log10_value', group_cols=['country', 'pathogen', 'site_id'])
        .pipe(add_rolling_features, value_col='log10_value', group_cols=['country', 'pathogen', 'site_id'], window=3)
    )
    display(clean_df.head())

    coverage = (
        clean_df.groupby(['country', 'pathogen'], dropna=False)
        .agg(start_date=('date', 'min'), end_date=('date', 'max'), n_observations=('date', 'size'), n_sites=('site_id', 'nunique'))
        .reset_index()
    )
    display(coverage)
    display(latest_signal_summary(clean_df))

    clean_df.to_parquet(PROCESSED / 'wastewater_long.parquet', index=False)
    clean_df.to_csv(PROCESSED / 'wastewater_long.csv', index=False)
    print(f'Saved {len(clean_df):,} rows to {PROCESSED}')

## 10. Plotting

Run after `clean_df` is populated and feature-engineered.

In [ ]:
if not clean_df.empty:
    for pathogen in sorted(clean_df['pathogen'].dropna().astype(str).unique()):
        fig, ax = plot_country_pathogen_timeseries(clean_df, pathogen=pathogen)
        plt.show()